In [1]:
import polars as pl
import points_dict
from tournament_dict import tournaments

In [26]:
# read file
file_2024 = 'atp_matches_2024.csv'
df = pl.read_csv(file_2024, infer_schema_length=None)
print(df.shape) # (3076, 49)

(3076, 49)


In [ ]:
keep_cols = ['tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'winner_id', 'winner_name', 'winner_seed', 'loser_id', 'loser_name', 'loser_seed', 'round']
winner_cols_rename = {'winner_id':'player_id', 'winner_name':'player_name', 'winner_seed':'player_seed'} 
non_winner_cols_rename = {'loser_id':'player_id', 'loser_name':'player_name', 'loser_seed':'player_seed'}

masters_1000_draw_size = {96: 128, 56:64}

df1 = df[keep_cols].clone()
df1.shape

(3076, 12)

In [31]:
def get_points(df, tourney_level, draw_size, points_dict):
    df_filter = df.filter((pl.col('tourney_level') == tourney_level) & (pl.col('draw_size') == draw_size))

    df_non_winner = df_filter.with_columns(pl.col('round').replace_strict(points_dict, default=None).alias('points'))
    df_non_winner = df_non_winner[['tourney_name','tourney_level','loser_id','loser_name','loser_seed','round','points']].rename(non_winner_cols_rename)

    df_winner = (
        df_filter
        .filter(pl.col('round') == 'F')
        .with_columns([
            pl.lit('W').alias('round'),
            pl.lit(points_dict['W']).alias('points')
        ])
    )

    df_winner = df_winner[['tourney_name','tourney_level','winner_id','winner_name','winner_seed','round','points']].rename(winner_cols_rename)
    df_winner

    df_non_winner = df_non_winner.with_columns(pl.col('points').cast(pl.Int32))
    df_points = pl.concat([df_non_winner, df_winner])
    df_points

    return df_points

In [ ]:
# df_gs = df1.filter(pl.col('tourney_level')=='G')

# df_gs_non_winner = df_gs.with_columns(pl.col('round').replace_strict(points_dict.points_GS, default=None).alias('points'))
# df_gs_non_winner = df_gs_non_winner[['tourney_name','tourney_level','loser_id','loser_name','points']].rename(non_winner_cols_rename)

# df_gs_winner = (
#     df_gs
#     .filter(pl.col('round') == 'F')
#     .with_columns([
#         pl.lit('W').alias('round'),
#         pl.lit(points_dict.points_GS['W']).alias('points')
#     ])
# )
# df_gs_winner = df_gs_winner[['tourney_name','tourney_level','winner_id','winner_name','points']].rename(winner_cols_rename)
# df_gs_winner

# df_gs_non_winner = df_gs_non_winner.with_columns(pl.col('points').cast(pl.Int32))
# df_gs_points = pl.concat([df_gs_non_winner, df_gs_winner])
# df_gs_points

tourney_name,tourney_level,player_id,player_name,points
str,str,i64,str,i32
"""Australian Open""","""G""",209976,"""Dino Prizmic""",10
"""Australian Open""","""G""",117360,"""Marc Polmans""",10
"""Australian Open""","""G""",105870,"""Yannick Hanfmann""",10
"""Australian Open""","""G""",104918,"""Andy Murray""",10
"""Australian Open""","""G""",104527,"""Stan Wawrinka""",10
…,…,…,…,…
"""Us Open""","""G""",126203,"""Taylor Fritz""",1300
"""Australian Open""","""G""",206173,"""Jannik Sinner""",2000
"""Roland Garros""","""G""",207989,"""Carlos Alcaraz""",2000


In [ ]:
### Grand Slams ###

df_gs = get_points(df1, 'G', 128, points_dict.points_GS)

df_gs_points = df_gs.pivot(
    values = "points",
    index = ["player_id", "player_name"],
    on = ["tourney_name", "tourney_level"],
    aggregate_function="first"
)

df_gs_points.filter(pl.col("player_name").is_in(['Carlos Alcaraz','Jannik Sinner','Novak Djokovic']))

player_id,player_name,"{""Australian Open"",""G""}","{""Roland Garros"",""G""}","{""Wimbledon"",""G""}","{""Us Open"",""G""}"
i64,str,i32,i32,i32,i32
207989,"""Carlos Alcaraz""",400,2000,2000,50
104925,"""Novak Djokovic""",800,400,1300,100
206173,"""Jannik Sinner""",2000,800,400,2000


In [79]:
### ATP Masters ###

df_masters_96 = get_points(df1, 'M', 128, points_dict.points_1000_96) # this is correct
df_masters_56 = get_points(df1, 'M', 56, points_dict.points_1000_56)


print(df_masters_96.shape, df_masters_56.shape)
print(df1.filter((pl.col('tourney_level')=='M') & (pl.col('draw_size')==128)).shape)

df1.filter((pl.col('tourney_level')=='M'))['tourney_name','draw_size'].unique()


#580

(384, 7) (56, 7)
(380, 12)


tourney_name,draw_size
str,i64
"""Indian Wells Masters""",128
"""Madrid Masters""",128
"""Shanghai Masters""",96
"""Cincinnati Masters""",64
"""Paris Masters""",56
"""Canada Masters""",64
"""Monte Carlo Masters""",64
"""Miami Masters""",128
"""Rome Masters""",128


In [ ]:
### Map 250 and 500 level tournaments ###
df_atp = df1.filter((pl.col('tourney_level')=='A') & (~pl.col('tourney_name').is_in(['United Cup', 'Laver Cup'])))

df_atp = df_atp.with_columns(pl.col("tourney_name").replace_strict(tournaments).alias('tourney_info'))

df_atp = df_atp.with_columns(
    pl.col("tourney_info").list.get(0).alias("tourney_level"),
    pl.col("tourney_info").list.get(1).alias("tourney_name")
)



(1081, 13) (404, 13)


In [ ]:
### ATP 500 ###

df_500_32 = get_points(df_atp, '500', 32, points_dict.points_500_32)
df_500_48 = get_points(df_atp, '500', 64, points_dict.points_500_48)
# Players that received a bye in the R32 in a 48 player Draw get 0 points

df_500_48 = df_500_48.with_columns(
   pl.when((pl.col('round')=='R32') & (pl.col('player_seed').is_not_null()))
     .then(pl.lit(0))
     .otherwise(pl.col('points'))
     .alias("points")
)

df_500_48.filter((pl.col('round')=='R32') & (pl.col('player_seed').is_not_null()))


tourney_name,tourney_level,player_id,player_name,player_seed,round,points
str,str,i64,str,i64,str,i32
"""Barcelona Open""","""500""",106432,"""Borna Coric""",15,"""R32""",0
"""Barcelona Open""","""500""",207518,"""Lorenzo Musetti""",10,"""R32""",0
"""Barcelona Open""","""500""",200005,"""Ugo Humbert""",6,"""R32""",0
"""Barcelona Open""","""500""",202104,"""Sebastian Baez""",8,"""R32""",0
"""Barcelona Open""","""500""",111797,"""Nicolas Jarry""",9,"""R32""",0
"""Barcelona Open""","""500""",126094,"""Andrey Rublev""",2,"""R32""",0
"""Washington Open""","""500""",106148,"""Roberto Carballes Baena""",11,"""R32""",0
"""Washington Open""","""500""",126846,"""Aleksandar Vukic""",14,"""R32""",0
"""Washington Open""","""500""",111575,"""Karen Khachanov""",3,"""R32""",0


In [58]:
### ATP 250 ###
df_250_32 = get_points(df_atp, '250', 32, points_dict.points_250_32)
df_250_48 = get_points(df_atp, '250', 64, points_dict.points_250_48)
# Players that received a bye in the R32 in a 48 player Draw get 0 points

df_250_48 = df_250_48.with_columns(
   pl.when((pl.col('round')=='R32') & (pl.col('player_seed').is_not_null()))
     .then(pl.lit(0))
     .otherwise(pl.col('points'))
     .alias("points")
)

df_250_48.filter((pl.col('round')=='R32') & (pl.col('player_seed').is_not_null()))

tourney_name,tourney_level,player_id,player_name,player_seed,round,points
str,str,i64,str,i64,str,i32
"""Winston-Salem Open""","""250""",202104,"""Sebastian Baez""",1,"""R32""",0
"""Winston-Salem Open""","""250""",209260,"""Luciano Darderi""",5,"""R32""",0
"""Winston-Salem Open""","""250""",202103,"""Francisco Cerundolo""",3,"""R32""",0
"""Winston-Salem Open""","""250""",208363,"""Mariano Navone""",7,"""R32""",0
"""Winston-Salem Open""","""250""",132686,"""Nuno Borges""",8,"""R32""",0
"""Winston-Salem Open""","""250""",206681,"""Fabian Marozsan""",9,"""R32""",0
"""Winston-Salem Open""","""250""",207686,"""Alexander Shevchenko""",13,"""R32""",0
"""Winston-Salem Open""","""250""",105173,"""Adrian Mannarino""",4,"""R32""",0
"""Winston-Salem Open""","""250""",144869,"""Tomas Martin Etcheverry""",6,"""R32""",0


In [ ]:
### ATP 500 - 64 Draw ###
df_atp_500_64 = df_atp_500.filter(pl.col('draw_size')==64).with_columns(pl.col("round").replace_strict(points_dict.points_500_32, default=None).alias("points"))
